## Rectangle size related problems

In [2]:
from typing import List

<b> 1727 largest submatrix with rearrangement </b>

the explanation from this [post](https://leetcode.com/problems/largest-submatrix-with-rearrangements/solutions/1020710/c-clean-and-clear-with-intuitive-pictures-o-m-n-logn/) is good

using a single list to store the height for each column when looping over each row, reset the height to 0 when the position is 0 no longer 1. After each row, sort the height collected, and calcuate the rectangle value collected. move forward until finish the loop

time complexity
- O(N*M) loop over the matrix, even the column M is looped twice

In [3]:

def largestSubmatrix(matrix: List[List[int]]) -> int:
    n = len(matrix)
    m = len(matrix[0])

    height = [0]*m # same size of the column number
    res = 0
    for i in range(n):
        for j in range(m):
            if matrix[i][j] == 1:
                height[j] += 1
            else:
                height[j] = 0

        sorted_height = sorted(height)
        for j in range(m):
            # rectangle size: hight * length
            # height is the sorted_height[j]
            # length is related to the position j
            res = max(res, sorted_height[j] * (m-j))
    return res


In [4]:
largestSubmatrix([[0,0,1],[1,1,1],[1,0,1]])

4

In [5]:
largestSubmatrix([[0,0,1],[1,1,1],[0,0,0]])

3

<b> 221 maximal square </b>

this [post](https://leetcode.com/problems/maximal-square/solutions/600149/python-thinking-process-diagrams-dp-approach/) explains the dynamic programming way very well.

to construct a square, its up, left, upleft position should all be 1 as well, otherwise the buidling tile break. The dp matrix is used to store the largest side value each position can get, and at the end the largest side value determines the largest squre size.

a smart way to construct the dp matrix with extra row and col than the original matrix, with default value 0.

In [6]:

def maximalSquare(matrix: List[List[str]]) -> int:
    if not matrix or not matrix[0]:
        return 0
    n = len(matrix)
    m = len(matrix[0])
    # create a matrix with size (n+1, m+1) to store the value of largest square side
    # each position can found
    # dp has extra row and col than matrix, assume matrix[i][j] correspond to dp[i+1][j+1]
    dp = [[0]*(m+1) for _ in range(n+1)] 
    largest_side = 0
    # loop over matrix to update the dp and keep track of the largest square side
    for i in range(n):
        for j in range(m):
            if matrix[i][j] == "1":
                # update dp by checking the up, left, upleft position
                # a square holds only when its up,left,upleft are non 0
                dp[i+1][j+1] = min(dp[i][j+1], dp[i+1][j], dp[i][j]) + 1
                largest_side = max(largest_side, dp[i+1][j+1])

    return largest_side*largest_side


<b> 85 maximal rectangle </b>

it is no longer a square, but a rectangle. it is a dp problems that should keep track of the left border position, right border position, height, when looping over each position of the matrix
- size of rectangle = (right - left) * height
- loop over each row
- left(i, j) = max(left(i-1, j), cur_left)
- right(i, j) = max(right(i-1, j), cur_right)
- height(i, j) = height(i,j)+1 if matrix[i][j] == 1 else 0

In [ ]:

def maximalRectangle(matrix: List[List[str]]) -> int:
    if not matrix or not matrix[0]:
        return 0
    n = len(matrix)
    m = len(matrix[0])
    """
    size of rectangle = (right - left) * height
    loop over each row
    left(i, j) = max(left(i-1, j), cur_left)
    right(i, j) = max(right(i-1, j), cur_right)
    height(i, j) = height(i,j)+1 if matrix[i][j] == 1 else 0
    """
    left = [0] * m # start from very left position 0
    right = [m] * m # start from very right position m
    height = [0] * m # initial height is 0
    res = 0

    for i in range(n):
        cur_left, cur_right = 0, m
        for j in range(m):
            if matrix[i][j] == "1":
                # if further right left border is the final left border
                left[j] = max(left[j], cur_left)
                height[j] += 1
            else:
                # position that left side of rectangle can start should move 1 step further
                # because current position is not qualified
                left[j] = 0 # => this value wont impact the rectangle size calculation when [i]j]==0, because the height will be reset to 0, so area - 0
                cur_left = j+1
                # height count starts over
                height[j] = 0
        for j in range(m-1, -1, -1):
            if matrix[i][j] == "1":
                # further left border is the final right border, so get the min val
                right[j] = min(right[j], cur_right)
            else:
                right[j] = m # back to initial state
                cur_right = j
        for j in range(m):
            # calculate the rectangle size
            res = max(res, (right[j]-left[j])*height[j])
    return res




<b> 84 largest rectangle in histogram </b>
for ith position, using the heights[i] to calculate the size. the max rectangle size is determined by the width it can get: r - l -1, where r represents the rightest location with h[r] >= h[i], because if h[r] < h[i], position r should NOT be considered as part of the rectangle. same, l represents the leftest location with h[l] >= h[i], if h[l]<h[i], position l should NOT be considered as well

so each i, we should know its right most and left most boundary that can use. h[i] as the height to calculate the rectangle size.

It is a smart way to determine the left_most[i] and right_most[i]

In [16]:

def largestRectangleArea(heights: List[int]) -> int:
    """
    for ith position, using the heights[i] to calculate the size
    the max rectangle size is determined by the width it can get: r - l -1
    where r represents the rightest location with h[r] >= h[i]
    because if h[r] < h[i], position r should NOT be considered as part of the rectangle
    same, l represents the leftest location with h[l] >= h[i]
    if h[l]<h[i], position l should NOT be considered as well

    so each i, we should know its right most and left most boundary that can use
    h[i] as the height to calculate the rectangle size
    """
    if not heights:
        return 0
    n = len(heights)
    max_area = 0
    # left_most[i] mean for ith position, the left most index position can reach
    left_most = [0] * n # initial position for left
    right_most = [0] * n # initial position for right
    left_most[0] = -1
    right_most[-1] = n
    for i in range(1, n):
        p = i-1
        while p >=0 and heights[p] >=heights[i]:
            # for p position, left_most[p] is the first left border where heights[left_most[p]] < heights[p]
            # heights[left_most[p]+1] >= heights[p]
            p = left_most[p]
        left_most[i] = p
    for i in range(n-2, -1, -1):
        p = i+1 # start from n-1
        while p < n and heights[p] >= heights[i]:
            p = right_most[p]
        right_most[i] = p

    for i in range(n):
        max_area = max(max_area, heights[i] * (right_most[i]-left_most[i]-1))
    return max_area

In [17]:
largestRectangleArea([0,9])

9

<b>764 largest plus sign </b>

also using a dp solution. The original grid matrix is not given but can be constructed by creating a (n,n) matrix with 1 as initialized value and mark position at mines with value 0.

However we need a matrix to keep track of the smallest count of 1 that each position can get after checking its four directions. this dp matrix can be combined with the original matrix, that is why the grid matrix is intialized to have value n at each position, because n is the largest number of count 1 that a position can have from 4 directions. for position from mines, the count of 1 should be 0 as well.

the for loop check the 4 direction of every positions, and update the smallest count of 1 one position can get from 4 directions, that is the size of plus sign one position can construct with.

In [7]:

def orderOfLargestPlusSign(n: int, mines: List[List[int]]) -> int:
    # create a dp matrix grid with size (n, n) to save the smallest count of 1
    # each position can get from checking its 4 directions
    # grid[i][j] means at position (i,j), check its 4 directions get the adjacent 1 count
    # then get the smallest count of 1 from the 4 directions.

    # initialize the grid, only position at mines are 0
    # n is the largest count of 1 a position can get, 
    # here combine the original matrix of 1, 0 to the count matrix
    grid = [[n]*n for _ in range(n)]
    for x,y in mines:
        grid[x][y] = 0

    for i in range(n):
        left, right, up, down = 0, 0, 0, 0

        for j in range(n):
            # loop over each column per row from left to right
            left = left + 1 if grid[i][j]!=0 else 0
            grid[i][j] = min(grid[i][j], left)

            # loop over each row per column from top to bottom
            # here j is the row count, i is the column index
            up = up + 1 if grid[j][i]!=0 else 0
            grid[j][i] = min(grid[j][i], up)
        for k in range(n-1, -1, -1):
            # loop over each column per row from right to left
            right = right + 1 if grid[i][k]!=0 else 0
            grid[i][k] = min(grid[i][k], right)

            # loop over each row per column from bottom to up
            # here k is the row count, i is the column index
            down = down + 1 if grid[k][i]!=0 else 0
            grid[k][i] = min(grid[k][i], down)

    # now every position is updated to have the smallest count of 1 it can get from 4 directions
    # the largest number from the grid is the answer
    res = 0
    for i in range(n):
        for j in range(n):
            res = max(res, grid[i][j])
    return res



In [8]:
orderOfLargestPlusSign(5, [[4,2]])

2

<b> 2201 Count Artifacts That Can Be Extracted </b>

way 1: using a matrix with size (n, n) to 
- first mark the cell for each artifact with unique index
- for each digged cell, set the value to 0 
- loop over the grid to find any cell with value > 0 meaning the artifact marked in the cell is not fully uncovered
- get the count of convered artifact

In [9]:

def digArtifacts(n: int, artifacts: List[List[int]], dig: List[List[int]]) -> int:
    # create a grid with size (n,n) with initial value 0 
    grid = [[0]*n for _ in range(n)]

    for ind, (start_i, start_j, end_i, end_j) in enumerate(artifacts):
        # mark each cell belongs to ind th artifacts with value ind
        # assuming each artifact only covers at most 4 cells
        for i in range(start_i, end_i+1):
            for j in range(start_j, end_j+1):
                grid[i][j] = ind+1 # starting with 1 because 0 is already used

    # suppose each artifact is extracted
    result = [1]*len(artifacts)
    for i, j in dig:
        grid[i][j] = 0 # mark as digged

    # check each cell of the grid and see if it has non 0 value that means the ind th artifact is not fully uncovered
    for i in range(n):
        for j in range(n):
            if grid[i][j] != 0:
                # the value of grid[i][j] - 1 is the ind of the artifact
                result[grid[i][j] - 1] = 0

    return sum(result)



In [12]:
digArtifacts(2, [[0,0,0,0],[0,1,1,1]], [[0,0],[0,1]])

1

way 2: using a set
- loop over each artifacts, collect all (i,j) belongs to the artifact,
- check if all those position belong to one artifact are in the dig set 
- if yes one artifact is uncovered

In [13]:

def digArtifacts(n: int, artifacts: List[List[int]], dig: List[List[int]]) -> int:
    dig = set((i, j) for i, j in dig)
    count = 0
    # check each artifact
    for start_i, start_j, end_i, end_j in artifacts:
        positions = set() # belongs to one artifact
        for i in range(start_i, end_i+1):
            for j in range(start_j, end_j+1):
                positions.add((i, j))
        # check if the positions are all digged by the dig set
        if all([pos in dig for pos in positions]):
            count += 1
    return count

<b> 2943 Maximize Area of Square Hole in Grid </b>

somewhat similar to previous rectangle problems. here checking if removing a bar will increase the square side or not, and the min value of the (largest_side_from_removing_hbar, largest_side_from_removing_vbar) is the side of the square

In [14]:
def maximizeSquareHoleArea(n: int, m: int, hBars: List[int], vBars: List[int]) -> int:

    def helper(bars):
        bars.sort()
        count = 2 # bar always has value, nonempty, so if removing a bar, the min count is 2
        res = 2
        for i in range(1, len(bars)):
            count = count + 1 if bars[i-1]+1==bars[i] else 2 # similar to if matrix[i][j]=0 height =0 from other problem
            res = max(res, count)
        return res

    side = min(helper(hBars), helper(vBars))
    return side * side

<b> 1793 maximum score of a good subarray </b>

start with i = j = k, score = A[k].
When increment the size of window, we want to reduce the min(A[i]..A[j]) slowly.

To do this, we can check the values on both sides of the window.  
If A[i - 1] < A[j + 1], we do j = j + 1  
If A[i - 1] >= A[j + 1], we do i = i - 1  


time complexity
- O(N)

In [15]:

def maximumScore( nums: List[int], k: int) -> int:
    """
    starting with position nums[k]
    determine to do j+1 or i+1
    since each time one element is added to the subarray, the min vlaue can be determined
    by checking the saved min value and the newly added item
    """
    res = nums[k]
    min_val = nums[k]
    i, j = k, k
    n = len(nums)
    while i > 0 or j < n-1:
        if i == 0:
            # hit the edge of i, only increase j
            j += 1
        elif j == n-1:
            i -= 1
        elif nums[i-1] < nums[j+1]:
            # adding nums[j+1] (the larger one) to the subarray can potentially increase the res
            j+=1
        else:
            i-=1

        min_val = min([min_val, nums[i], nums[j]])
        res = max(res, min_val * (j-i+1))
    return res
